In [2]:
# ДОМАШНЯЯ РАБОТА №2
# Сравнение AutoML-фреймворков (H2O vs FLAML)
# Задача: Прогнозирование доклинического риска
# Карташова Ольга, группа 737-01

# Установила библиотеки
!pip install h2o flaml pandas scikit-learn -q

print("Библиотеки установлены!")

Библиотеки установлены!


In [3]:
# Импортирую библиотеки

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import h2o
from h2o.automl import H2OAutoML
from flaml import AutoML
import requests
import io

print(" Библиотеки импортированы!")

 Библиотеки импортированы!


                             import pandas as pd.
Pandas — это работа с таблицами (как Excel в Python). Позволяет читать CSV-файлы, смотреть данные, удалять лишние колонки, фильтровать строки.

Что делает в коде:
*   pd.read_csv() — загружает файл с данными
*   df.head() — показывает первые строки (если добавить)
*   df.drop() — удаляет ненужные колонки


              from sklearn.model_selection import train_test_split.
Часть библиотеки scikit-learn (набор инструментов для машинного обучения).Делит данные на две части: обучающую (на которой модель учится) и тестовую (на которой проверяем, как хорошо модель научилась).

Что делает в коде:

*   train_test_split() — разбивает данные на тренировочные (80%) и тестовые (20%).

Пример: Как в школе: есть задачи для тренировки (домашнее задание) и задачи для проверки (контрольная работа). Модель учится на тренировочных, а проверяем на тестовых.


                   from sklearn.metrics import accuracy_score.
Тоже часть scikit-learn. Содержит метрики для оценки качества моделей. Считает, насколько часто модель угадала правильный ответ.

Что делает в коде:
*   accuracy_score(правильные_ответы, предсказания_модели) — возвращает долю правильных ответов (от 0 до 1).

Пример: Если модель правильно предсказала 85 из 100 пациентов, точность = 0.85 (85%).


                                   import h2o.

H2O — это платформа для машинного обучения, которая работает на Java. Позволяет запускать H2O AutoML — автоматический подбор лучшей модели.

Что делает в коде:


*   h2o.init() — запускает сервер H2O на вашем компьютере
*   h2o.H2OFrame() — превращает pandas-таблицу в формат, понятный H2O
*   h2o.cluster().shutdown() — останавливает сервер

Важно: Для работы H2O нужна установленная Java (это отдельная программа). Если Java нет — H2O не запустится.


                           from flaml import AutoML.
FLAML (Fast and Lightweight AutoML) — фреймворк от Microsoft для автоматического машинного обучения. То же самое, что H2O AutoML, но легче и быстрее. Не требует Java.

Что делает в коде:


*   AutoML() — создаёт объект для автоматического обучения
*   .fit() — запускает обучение на ваших данных
*   .predict() — предсказывает результаты на новых данных
*   .best_estimator — показывает, какую модель выбрал фреймворк

Пример: Это «конкурент» H2O, но от Microsoft. Как Coca-Cola и Pepsi — делают одно и то же, но по-разному.


                                import requests
Библиотека для скачивания файлов из интернета. Позволяет загрузить CSV-файл прямо из GitHub, не скачивая его на компьютер руками.

Что делает в коде:
*   requests.get(url) — скачивает файл по ссылке. Даем ссылку, а requests скачивает содержимое.


                                  import io
Стандартная библиотека Python для работы с данными в памяти. Превращает скачанный текст в формат, который понимает pandas.

Что делает в коде:

*   io.StringIO() — берёт текст из интернета и делает из него «виртуальный файл», который можно прочитать через pd.read_csv().

Пример: Как буфер обмена: скопировали текст, а io.StringIO превращает его в файл.

In [7]:
# Читаем CSV файл в pandas DataFrame

print("ЗАГРУЗКА ДАННЫХ")

# Загружаем данные из репозитория (с немного другими данными)
url = "https://raw.githubusercontent.com/KateAndri/StudyRepo26/main/data/dispensarization_data_2026.csv"
response = requests.get(url)
df = pd.read_csv(io.StringIO(response.content.decode('utf-8')))

print(f"   Данные загружены!")
print(f"   Количество записей (пациентов): {len(df)}")
print(f"   Количество признаков: {len(df.columns)}")
print(f"   Доступные колонки: {list(df.columns)}")

ЗАГРУЗКА ДАННЫХ
   Данные загружены!
   Количество записей (пациентов): 1000
   Количество признаков: 18
   Доступные колонки: ['Возраст', 'Пол_мужской', 'ИМТ', 'Окружность_талии_см', 'САД_мм_рт_ст', 'ДАД_мм_рт_ст', 'Пульсовое_давление', 'Глюкоза_натощак_ммоль_л', 'HbA1c_%', 'ЛПНП_ммоль_л', 'ЛПВП_ммоль_л', 'Триглицериды_ммоль_л', 'СКФ_мл_мин', 'Курение', 'Физическая_активность_мин_нед', 'ССЗ_риск_высокий', 'Статус_глюкозы', 'Доклинический_риск']


Весь этот блок кода — это ритуал загрузки данных. Указываем, где лежит файл (в интернете), скачиваем его, а затем превращаем в удобную табличку (DataFrame), с которой будет работать вся остальная программа.

В результате выполнения этого блока кода, в памяти программы появится переменная df. Я могу представить её в виде таблицы. Вся дальнейшая работа (чистка данных, обучение моделей) будет проводиться именно с этой таблицей df. Я превратили статичный CSV-файл в интернете в живой и готовый к исследованиям объект в программе.


In [8]:
# Определяем целевую переменную

print("ПОДГОТОВКА ДАННЫХ")

# Находим колонку с риском
target_col = None
for col in df.columns:
    if 'риск' in col.lower():
        target_col = col
        break

if target_col is None:
    print(" Ошибка: не найдена")
    print(f"   Доступные колонки: {list(df.columns)}")
    exit()

print(f" Целевая переменная: {target_col} ")
print(f"   Распределение классов:")
print(f"   {df[target_col].value_counts()}")



ПОДГОТОВКА ДАННЫХ
 Целевая переменная: ССЗ_риск_высокий 
   Распределение классов:
   ССЗ_риск_высокий
0    819
1    181
Name: count, dtype: int64


Я загрузила таблицу df с 1000 пациентами и 18 колонками. Но модель машинного обучения не может работать с «сырыми» данными. Их нужно подготовить:

*   Выбрать, что будем предсказывать — найти колонку-«ответ» (целевую переменную). В моем случае это Доклинический_риск.
*   Отделить ответ от признаков — создать X (входные данные) и y (правильные ответы).
*   Очистить данные от «мусора» — удалить текстовые колонки, которые модель не понимает.
*   Разделить данные на две части — на тех, на ком модель будет учиться (обучающая выборка), и тех, на ком я буду проверять её умения (тестовая выборка).


                            target = None
Создаём переменную target и пока кладём в неё «пустышку» (None). Она будет хранить имя колонки, которую мы будем предсказывать.

                          for col in df.columns:
Это цикл.

df.columns — это список всех названий колонок (18 штук). Цикл говорит: «Перебери все колонки по очереди, каждую назови col».

                          if 'риск' in col.lower():
Проверка условия.

                                   col.lower()
Берёт название колонки и переводит его в нижний регистр (маленькими буквами). Например, "Доклинический_риск" станет "доклинический_риск". Это делается, чтобы не зависеть от того, с большой или маленькой буквы написано слово «риск».

                                   'риск' in ...
Проверяет, есть ли слово «риск» внутри названия колонки.
Если есть — условие истинно.

                                   target = col
Если нашли колонку со словом «риск», сохраняем её имя в переменную target.

                                       break
Немедленно выходим из цикла. Зачем продолжать перебирать остальные колонки, если мы уже нашли то, что искали?

**Итог: Переменная target теперь хранит строку "Доклинический_риск".**


Проверка ошибки. Это защита от ошибок — программа сама себя проверяет и говорит, что пошло не так.

                              if target is None:

Если после цикла target так и остался пустым (None), значит, колонка со словом «риск» не найдена.

                                   exit()
Аварийно останавливает программу. Дальше работать бессмысленно.


                                 df[target]
Берём колонку с названием target (то есть Доклинический_риск). Это серия (одномерный массив) из 1000 нулей и единиц.

                                df[target] == 0  
Сравниваем каждое значение с 0. Получается массив из True и False. True там, где значение равно 0.

                                    .sum()
Суммируем эти значения. В Python True считается за 1, а False за 0. Так мы получаем количество пациентов, у которых нет риска.



In [9]:
# Делим данные: 80% на обучение, 20% на тестирование

train, test = train_test_split(df, test_size=0.2, random_state=42)
print(f" Обучающая выборка: {len(train)} пациентов")
print(f" Тестовая выборка: {len(test)} пациентов")

 Обучающая выборка: 800 пациентов
 Тестовая выборка: 200 пациентов


**Обучающая выборка (train)** — это домашние задания и упражнения, которые ученик (модель) решает во время учёбы. Он видит и вопросы, и правильные ответы, и учится на своих ошибках.

**Тестовая выборка (test)** — это контрольная работа, которую ученик пишет в конце четверти. Он не видел эти задания раньше. По тому, как он справился с ними, вы оцениваете, насколько хорошо он усвоил материал.

Пример: Если ученик увидит контрольную заранее — вы не узнаете его реальный уровень знаний. Так же и с моделью: если она увидит тестовые данные до обучения, вы получите «лживую» завышенную оценку.


**train**	Модель учится на этих данных (видит и вопросы, и ответы)

**test**	Модель проверяют на этих данных (видит только вопросы, ответы скрыты)

**test_size=0.2**	20% данных идёт на проверку, 80% — на обучение

**random_state=42**	Фиксирует случайность, чтобы результаты были одинаковыми при каждом запуске


In [11]:
# Запускаем H2O AutoML (как в коде преподавателя)

print("ОБУЧЕНИЕ H2O AutoML")

try:
    # Запускаем H2O
    h2o.init(verbose=False)

    # Конвертируем pandas DataFrame в H2O формат
    train_h2o = h2o.H2OFrame(train)
    test_h2o = h2o.H2OFrame(test)

    # Преобразуем целевую переменную в фактор для задачи классификации
    train_h2o[target_col] = train_h2o[target_col].asfactor()

    # Создаём AutoML модель: максимум 180 секунд, фиксируем случайность
    aml = H2OAutoML(max_runtime_secs=180, seed=42)

    # Запускаем обучение, указываем целевую переменную
    aml.train(y=target_col, training_frame=train_h2o)

    # Делаем предсказания на тестовых данных
    pred = aml.leader.predict(test_h2o)
    # Считаем точность
    h2o_acc = accuracy_score(test[target_col], pred['predict'].as_data_frame().values.flatten()[:len(test)])
    print(f" H2O: {h2o_acc:.4f} - {aml.leader.model_id}")

    # Останавливаем H2O
    h2o.cluster().shutdown()

except Exception as e:
    print(f" H2O не запустился (требуется Java): {e}")
    print("   Пропускаем H2O, продолжаем с FLAML")
    h2o_acc = None

ОБУЧЕНИЕ H2O AutoML
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


 H2O: 0.9950 - StackedEnsemble_BestOfFamily_4_AutoML_1_20260513_193432


*   **h2o.init()** - Запускает сервер H2O (как двигатель)
*   **h2o.H2OFrame()** - Превращает ваши данные в формат, понятный H2O
*   **.asfactor()** - Говорит: «Это ответ — да/нет, а не число»
*   **H2OAutoML(max_runtime_secs=180)** - Создаёт автоматическую модель, которая будет работать 3 минуты
*   **.train()** - Запускает обучение — H2O сам пробует разные алгоритмы
*   **.predict()** - Предсказывает ответы для тестовых пациентов
*   **accuracy_score()** -Сравнивает предсказания с реальными ответами (считает точность)
*   **h2o.cluster().shutdown()** - Выключает сервер H2O (экономит память)


0.9950 — точность модели (99.5 % правильных ответов).

StackedEnsemble — H2O выбрал комбинацию нескольких моделей.







In [12]:
# Отделяем признаки (X) от целевой переменной (y) для FLAML
X_train, y_train = train.drop(target_col, axis=1), train[target_col]
X_test, y_test = test.drop(target_col, axis=1), test[target_col]

# Удаляем текстовые колонки (если есть), как в примере преподавателя
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        print(f"   Удаляем текстовую колонку: {col}")
        X_train = X_train.drop(columns=[col])
        X_test = X_test.drop(columns=[col])

print(f" Признаки для FLAML: {list(X_train.columns)}")

 Признаки для FLAML: ['Возраст', 'Пол_мужской', 'ИМТ', 'Окружность_талии_см', 'САД_мм_рт_ст', 'ДАД_мм_рт_ст', 'Пульсовое_давление', 'Глюкоза_натощак_ммоль_л', 'HbA1c_%', 'ЛПНП_ммоль_л', 'ЛПВП_ммоль_л', 'Триглицериды_ммоль_л', 'СКФ_мл_мин', 'Курение', 'Физическая_активность_мин_нед', 'Статус_глюкозы', 'Доклинический_риск']


                        train.drop(target_col, axis=1)

Удаляет колонку-ответ из обучающей выборки	Оставляем только признаки (возраст, давление, холестерин...).

                             train[target_col]
Берёт только колонку-ответ	Это правильные ответы (есть риск / нет риска).

То же самое для test	Делает то же с тестовой выборкой	Подготавливаем данные для проверки.

                         Цикл for col in X_train.columns
Проверяет каждую колонку.	Ищем текстовые колонки.

                               if dtype == 'object'
Нашёл текстовую колонку	Например, «ФИО» или «Адрес».

                            X_train.drop(columns=[col])
Удаляет эту колонку	Модель не понимает текст, лучше убрать.


**Итог:** После этой подготовки X_train и y_train пойдут в FLAML на обучение, а X_test и y_test — на проверку.

In [13]:
# Обучаем FLAML
print("ОБУЧЕНИЕ FLAML")

try:
    # Создаём AutoML модель
    automl = AutoML()

    # Запускаем обучение:
    # - task='classification' - задача классификации
    # - time_budget=180 - максимум 180 секунд (как у преподавателя)
    automl.fit(X_train, y_train, task='classification', time_budget=180)

    # Считаем точность
    flaml_acc = accuracy_score(y_test, automl.predict(X_test))
    print(f" FLAML: {flaml_acc:.4f} - {automl.best_estimator}")

except Exception as e:
    print(f" Ошибка FLAML: {e}")
    flaml_acc = None


ОБУЧЕНИЕ FLAML
[flaml.automl.logger: 05-13 19:45:50] {2375} INFO - task = classification
[flaml.automl.logger: 05-13 19:45:50] {2386} INFO - Evaluation method: cv
[flaml.automl.logger: 05-13 19:45:50] {2489} INFO - Minimizing error metric: 1-roc_auc
[flaml.automl.logger: 05-13 19:45:50] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 05-13 19:45:50] {2911} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 05-13 19:45:50] {3046} INFO - Estimated sufficient time budget=1961s. Estimated necessary time budget=45s.
[flaml.automl.logger: 05-13 19:45:50] {3097} INFO -  at 0.2s,	estimator lgbm's best error=4.8030e-02,	best estimator lgbm's best error=4.8030e-02
[flaml.automl.logger: 05-13 19:45:50] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 05-13 19:45:50] {3097} INFO -  at 0.3s,	estimator lgbm's best error=3.2206e-02,	best estimator lgbm's best error=3.2206e-

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 05-13 19:45:51] {3097} INFO -  at 1.1s,	estimator sgd's best error=2.6998e-01,	best estimator lgbm's best error=9.5627e-03
[flaml.automl.logger: 05-13 19:45:51] {2911} INFO - iteration 10, current learner sgd
[flaml.automl.logger: 05-13 19:45:51] {3097} INFO -  at 1.3s,	estimator sgd's best error=2.2331e-01,	best estimator lgbm's best error=9.5627e-03
[flaml.automl.logger: 05-13 19:45:51] {2911} INFO - iteration 11, current learner lgbm
[flaml.automl.logger: 05-13 19:45:51] {3097} INFO -  at 1.3s,	estimator lgbm's best error=4.7747e-03,	best estimator lgbm's best error=4.7747e-03
[flaml.automl.logger: 05-13 19:45:51] {2911} INFO - iteration 12, current learner xgboost
[flaml.automl.logger: 05-13 19:45:52] {3097} INFO -  at 1.5s,	estimator xgboost's best error=4.8066e-02,	best estimator lgbm's best error=4.7747e-03
[flaml.automl.logger: 05-13 19:45:52] {2911} INFO - iteration 13, current learner lgbm
[flaml.automl.logger: 05-13 19:45:52] {3097} INFO -  at 1.6s,	est

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 05-13 19:48:49] {3097} INFO -  at 179.4s,	estimator lrl1's best error=1.4282e-01,	best estimator lgbm's best error=0.0000e+00
[flaml.automl.logger: 05-13 19:48:49] {2911} INFO - iteration 334, current learner lrl1
[flaml.automl.logger: 05-13 19:48:50] {3097} INFO -  at 179.8s,	estimator lrl1's best error=1.4282e-01,	best estimator lgbm's best error=0.0000e+00
[flaml.automl.logger: 05-13 19:48:50] {2911} INFO - iteration 335, current learner xgboost
[flaml.automl.logger: 05-13 19:48:50] {3097} INFO -  at 180.0s,	estimator xgboost's best error=4.2234e-03,	best estimator lgbm's best error=0.0000e+00
[flaml.automl.logger: 05-13 19:48:50] {3359} INFO - retrain lgbm for 0.1s
[flaml.automl.logger: 05-13 19:48:50] {3362} INFO - retrained model: LGBMClassifier(colsample_bytree=np.float64(0.9119902181768895),
               learning_rate=np.float64(0.42067294230498364), max_bin=1023,
               min_child_samples=11, n_estimators=220, n_jobs=-1, num_leaves=4,
           

                             automl = AutoML()
Создаёт объект для автоматического обучения	Готовим «робота-учителя».

        automl.fit(X_train, y_train, task='classification', time_budget=180)
Запускает обучение	Робот пробует разные алгоритмы 3 минуты.

                          automl.predict(X_test)
Предсказывает ответы для тестовых пациентов	Проверяем, чему научился.

                         accuracy_score(y_test, ...)
Сравнивает предсказания с реальностью	Считает долю правильных ответов.

                                .best_estimator
Показывает, какой алгоритм выбрал FLAML	Например, «XGBoost» или «RandomForest».




**task='classification'**	- Задача классификации	Предсказываем «да/нет» (0 или 1)

**time_budget=180**	- 180 секунд (3 минуты)	FLAML сам остановится через 3 минуты


**.best_estimator** -	Лучший алгоритм	FLAML сам выбрал, например, XGBoost


                                        Итог
0.99 — точность модели (99% правильных ответов)

XGBoost — алгоритм, который FLAML выбрал как лучший



In [18]:
# ИТОГОВОЕ СРАВНЕНИЕ

print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ СРАВНЕНИЯ")

if h2o_acc is not None:
    print(f"H2O:   {h2o_acc:.4f}")
else:
    print("H2O:   не удалось запустить (требуется Java)")

if flaml_acc is not None:
    print(f"FLAML: {flaml_acc:.4f}")
else:
    print("FLAML: ошибка")

print("ВЫВОДЫ")

if h2o_acc is not None and flaml_acc is not None:
    if h2o_acc > flaml_acc:
        print(" Лучший результат показал H2O AutoML")
    elif flaml_acc > h2o_acc:
        print(" Лучший результат показал FLAML")
    else:
        print(" Оба фреймворка показали одинаковый результат")
elif flaml_acc is not None:
    print(" FLAML успешно обучился и показал точность: {:.4f}".format(flaml_acc))
    print("   (H2O не запустился из-за отсутствия Java)")




ИТОГОВЫЕ РЕЗУЛЬТАТЫ СРАВНЕНИЯ
H2O:   0.9950
FLAML: 0.9900
ВЫВОДЫ
 Лучший результат показал H2O AutoML


**if h2o_acc is not None:**	Проверяем, удалось ли запустить H2O

**print(f"H2O: {h2o_acc:.4f}")**	Выводим точность H2O (4 знака после запятой)


**if flaml_acc is not None:**	Проверяем, удалось ли запустить FLAML

**Сравнение if h2o_acc > flaml_acc**	Определяем, у кого точность выше

**print("Лучший: ...")**	Объявляем победителя


Я взяла за основу код из Вашего примера и адаптировала его под свою задачу — прогнозирование доклинического риска. Вместо данных о стадиях сна я использовала данные диспансеризации, вместо целевой переменной «Стадия» — «ССЗ_риск_высокий». Остальная логика (разделение на train/test, обучение H2O и FLAML, расчёт точности) осталась такой же, как в примере.

Код успешно сравнил H2O и FLAML: оба фреймворка показали высокую точность, но FLAML предпочтительнее для реального внедрения из-за простоты установки и возможности объяснять прогноз